# 静态 Kernel 编译功能

## 1. 功能简介

对于纯静态 shape 网络或者 shape 变化较少的动态 shape 网络，如需提升网络执行性能，可通过算子预先静态编译达到目的，该方式简称为静态 Kernel 编译。它是指在模型编译时指定 shape 大小，运行时不需要指定 shape 大小，减少运行时开销，具体优势如下：

- 编译时已知所有 Tensor 的大小，存储空间利用率高。
- 编译时可以针对实际的 shape 大小做针对性优化。
- AI 处理器擅长并行指令运行，不擅长逻辑计算。静态编译可以在编译时完成标量计算，减少 Scalar 操作对并行指令的打断，提升性能。
- 算子编译工具在编译时知道确切的数据大小，不会额外插入同步，避免并行指令变成串行执行。

开启静态 Kernel 编译后，系统会根据算子信息统计文件获取确定的 Shape，并为各组 Shape 生成对应的算子二进制。Shape 组合过多时会增加编译时间和产物规模，因此该能力更适合 Shape 稳定的模型。

## 2. 使用约束

- 本功能支持的产品型号参见[使用说明](https://gitcode.com/Ascend/torchair/blob/master/docs/zh/overview.md#%E4%BD%BF%E7%94%A8%E8%AF%B4%E6%98%8E)。
- 当 Ascend PyTorch Profiler 中 `experimental_config` 参数开启算子信息统计功能（即 `record_op_args=True`），且 `schedule` 参数的 `skip_first=0` 时，不支持同时使用本功能。
- 多卡场景必须配置环境变量 `LOCAL_WORLD_SIZE`（指单节点上启动的并行进程数），且每个节点的值必须一致。
  > 多卡场景在安装静态 kernel run 包时会更新算子库公共文件，若无 `LOCAL_WORLD_SIZE` 协同，可能读取到过程态内容导致未定义行为。

## 3. 使用方法

通过 `npugraph_ex` 的 `options` 配置开启：

<table align="left" border="1" cellpadding="6" cellspacing="0">
  <tr><th align="left">参数名</th><th align="left">说明</th></tr>
  <tr><td align="left"><code>static_kernel_compile</code></td><td align="left">是否开启静态 Kernel 编译。<code>False</code>（默认）：关闭；<code>True</code>：开启。产物默认位于当前执行脚本的同级目录；同时启用模型编译缓存且未启用 <code>force_eager</code> 时，产物位于模型缓存文件所在目录。</td></tr>
  <tr><td align="left"><code>disable_static_kernel_compile_cache</code></td><td align="left">是否禁用静态 Kernel 缓存。<code>False</code>（默认）：使用缓存；<code>True</code>：每次重新编译。仅在 <code>static_kernel_compile=True</code> 时生效。</td></tr>
</table>
<div style="clear: both;"></div>
> **多卡场景配置（可选）**：运行前需将 `LOCAL_WORLD_SIZE` 设置为单节点并行进程数，例如 `export LOCAL_WORLD_SIZE=${local_world_size}`。同一节点上的所有进程必须使用一致的值。


## 4. 使用示例

下面的示例展示如何开启静态 Kernel 编译。第一次运行会触发编译并生成产物，后续在硬件、CANN 版本、算子属性和相关选项一致时命中缓存，直接复用编译好的 Kernel。

In [ ]:
import os
import torch
import torch_npu


class StaticShapeModel(torch.nn.Module):
    def forward(self, x, weight):
        return torch.relu(torch.mm(x, weight))


# 多卡场景需要配置 LOCAL_WORLD_SIZE（单卡场景可省略）
os.environ.setdefault("LOCAL_WORLD_SIZE", "1")

# 固定 Shape 模型适合将更多工作前移到静态 Kernel 编译阶段。
model = StaticShapeModel().npu()
compiled = torch.compile(
    model,
    backend="npugraph_ex",
    options={
        # 开启静态 Kernel 编译
        "static_kernel_compile": True,
        # 使用缓存，二次运行命中时直接复用编译好的 kernel
        "disable_static_kernel_compile_cache": False,
    },
    dynamic=False,
    fullgraph=True,
)

x = torch.randn(128, 256, dtype=torch.float16).npu()
weight = torch.randn(256, 256, dtype=torch.float16).npu()

# 首次执行生成静态 Kernel 及缓存，条件一致时后续运行可复用缓存。
out = compiled(x, weight)
torch.npu.synchronize()
print("output:", tuple(out.shape))

## 5. 编译产物说明

编译产物默认在 `static_kernel_compile_outputs` 目录下。运行用户需要对进程工作目录具有读、写和执行权限，主要结构如下：

```
static_kernel_compile_outputs
├── static_kernel_cache          # 静态 Kernel 缓存目录
│   └── CANN-{version}_{device}.json   # 缓存匹配关系记录
└── ts{timestamp}_pid{pid}_outputs
    ├── {pid}                    # 目标算子信息（shape、format 等）
    ├── {pid}_opcompile           # 支持静态编译的算子信息
    └── static_kernel_{datetime}.run   # 编译好的静态 Kernel 文件
```

> 当静态 Kernel 编译进程异常退出时，请根据终端提示信息中的 `uninstall.sh` 脚本路径执行卸载。

## 6. 缓存命中条件

使用静态 Kernel 缓存时，系统会对比当前运行特征与缓存记录。以下条件全部满足且对应的 kernel run 包存在时，才判定为缓存命中：

1. CANN 版本不变、硬件型号不变；
2. 模型中所有参与静态 Kernel 编译的算子属性（类型、输入输出 Shape、Format、数据类型等）保持一致；
3. 静态编译时的特定选项保持一致（确定性计算配置、`super_kernel_optimize` 相关配置等）。

若任一条件不匹配，将触发完整的静态 Kernel 编译流程。

## 7. 课后练习

### 一、单选题

（1）【单选题】静态 Kernel 编译最适合哪类模型或输入场景？
- A. Shape 完全随机且不可枚举
- B. Shape 完全固定，或 Shape 种类少且可枚举
- C. 只能在 CPU 上运行的模型
- D. 必须包含随机数算子的模型

（2）【单选题】开启静态 Kernel 编译的 options 参数是？
- A. `static_kernel_compile=True`
- B. `force_recapture=True`
- C. `clone_output=True`
- D. `cache_compile=True`

（3）【单选题】在 `static_kernel_compile=True` 时，设置 `disable_static_kernel_compile_cache=False` 的含义是？
- A. 每次都重新编译
- B. 允许命中并复用静态 Kernel 缓存
- C. 删除已有缓存
- D. 禁止生成静态 Kernel

（4）【单选题】多卡场景使用静态 Kernel 编译前，必须配置哪个环境变量？
- A. CUDA_VISIBLE_DEVICES
- B. LOCAL_WORLD_SIZE
- C. PYTHONPATH
- D. OMP_NUM_THREADS

（5）【单选题】以下哪项会导致静态 Kernel 缓存不命中？
- A. 硬件型号变化
- B. 仅查看一次输出 Shape
- C. 增加一条打印日志
- D. 创建一个新的 Python 注释

（6）【单选题】当 Profiler 的 `record_op_args=True` 且 `skip_first=0` 时，对静态 Kernel 编译的影响是？
- A. 可以无条件同时使用
- B. 不支持同时使用
- C. 自动开启 SuperKernel
- D. 自动关闭动态 Shape

### 二、多选题

（7）【多选题】静态 Kernel 编译可能带来的收益包括哪些？
- A. 在编译阶段完成部分 Shape 推导和标量计算
- B. 根据确定的 Shape 做针对性优化
- C. 减少部分运行时同步与 Scalar 操作开销
- D. 让任意动态 Shape 都无需重新编译

（8）【多选题】静态 Kernel 缓存命中需要哪些条件保持一致？
- A. CANN 版本和硬件型号
- B. 参与编译算子的属性，如 Shape、Format、数据类型
- C. 确定性计算和 `super_kernel_optimize` 等相关选项
- D. Python 文件的注释行数

（9）【多选题】多卡场景使用静态 Kernel 编译时，正确的做法有哪些？
- A. 配置 LOCAL_WORLD_SIZE 为单节点并行进程数
- B. 保证每个节点的 LOCAL_WORLD_SIZE 值一致
- C. 注意安装静态 kernel run 包会更新公共算子库文件
- D. 完全不需要进程间协同

（10）【多选题】关于静态 Kernel 编译产物和缓存，正确的说法有哪些？
- A. 默认产物目录包含 `static_kernel_compile_outputs`
- B. 缓存目录中会记录 CANN 版本与设备相关的匹配关系
- C. 可设置 `disable_static_kernel_compile_cache=True` 强制每次重新编译
- D. 进程异常退出后，不需要处理任何安装残留

**运行以下代码单元查看参考答案与解析。**


In [ ]:
import os
answer_path = "answer/04.05_answer.txt"
if os.path.exists(answer_path):
    with open(answer_path, "r", encoding="utf-8") as f:
        print(f.read())
else:
    print("答案文件未找到，请检查 answer 目录。")
